# Fetch Weather Data — Shenzhen 2022

Descarga datos meteorológicos históricos de **Open-Meteo** (gratuito, sin API key) para Shenzhen, China durante el período del dataset ST-EVCDP: **19 Jun – 18 Jul 2022**.

- Fuente: [open-meteo.com/en/docs/historical-weather-api](https://open-meteo.com/en/docs/historical-weather-api)
- Variables: temperatura (°C), humedad relativa (%), precipitación (mm)
- Resolución original: **horaria** → interpolada a **5 minutos** (8640 timesteps) para coincidir con ST-EVCDP
- Output: `weather_shenzhen.csv`

Este CSV se carga en `ChatEV.ipynb` como feature adicional en los prompts (§3.1.1 del paper: *Weather conditions*).

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Coordenadas de Shenzhen (centro urbano)
LAT = 22.5431
LON = 114.0579

# Período exacto del dataset ST-EVCDP
START_DATE = "2022-06-19"
END_DATE   = "2022-07-18"

# Variables meteorológicas del paper (§3.1.1)
VARIABLES = "temperature_2m,relativehumidity_2m,precipitation"

url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={LAT}&longitude={LON}"
    f"&start_date={START_DATE}&end_date={END_DATE}"
    f"&hourly={VARIABLES}"
    f"&timezone=Asia%2FShanghai"
)

print(f"Fetching: {url[:100]}...")
resp = requests.get(url, timeout=30)
resp.raise_for_status()
data = resp.json()

print(f"Status: OK")
print(f"Keys: {list(data.keys())}")
print(f"Hourly rows: {len(data['hourly']['time'])}")

In [ ]:
# Construir DataFrame horario
hourly = data["hourly"]

df_hourly = pd.DataFrame({
    "timestamp":    pd.to_datetime(hourly["time"]),
    "temperature":  hourly["temperature_2m"],
    "humidity":     hourly["relativehumidity_2m"],
    "precipitation":hourly["precipitation"],
})

df_hourly = df_hourly.set_index("timestamp")

print(f"Hourly data shape: {df_hourly.shape}")
print(f"Range: {df_hourly.index[0]}  →  {df_hourly.index[-1]}")
print(f"\nStats:")
print(df_hourly.describe().round(2))

In [ ]:
# Interpolar de horario → 5 minutos (×12) para coincidir con ST-EVCDP
# ST-EVCDP: 8640 timesteps = 30 días × 288 pasos/día (cada 5 min)

TARGET_TIMESTEPS = 8640

df_5min = df_hourly.resample("5min").interpolate(method="linear")

# Recortar exactamente a 8640 filas
df_5min = df_5min.iloc[:TARGET_TIMESTEPS].copy()

# Si faltan filas (raro), rellenar con forward-fill
if len(df_5min) < TARGET_TIMESTEPS:
    extra = pd.DataFrame(
        index=pd.date_range(df_5min.index[-1], periods=TARGET_TIMESTEPS - len(df_5min) + 1, freq="5min")[1:],
        columns=df_5min.columns,
    )
    df_5min = pd.concat([df_5min, extra]).ffill()

# Resetear índice para guardar timestamp como columna
df_5min = df_5min.reset_index()
df_5min.columns = ["timestamp", "temperature", "humidity", "precipitation"]

print(f"5-min data shape: {df_5min.shape}  (target: {TARGET_TIMESTEPS} rows)")
print(f"NaN values: {df_5min.isnull().sum().sum()}")
print(f"\nMuestra (primeras 3 filas):")
print(df_5min.head(3).to_string())

In [ ]:
# Validación visual
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
fig.suptitle("Datos Meteorológicos — Shenzhen, Jun–Jul 2022 (5 min/paso)", fontsize=12)

n_show = 288 * 7  # primera semana

axes[0].plot(df_5min["temperature"][:n_show], color="#E74C3C", lw=0.8)
axes[0].set_ylabel("Temperatura (°C)")
axes[0].grid(alpha=0.3)

axes[1].plot(df_5min["humidity"][:n_show], color="#2980B9", lw=0.8)
axes[1].set_ylabel("Humedad relativa (%)")
axes[1].grid(alpha=0.3)

axes[2].fill_between(range(n_show), df_5min["precipitation"][:n_show], color="#1ABC9C", alpha=0.6)
axes[2].set_ylabel("Precipitación (mm)")
axes[2].set_xlabel("Timestep (5 min/paso)")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("weather_shenzhen_preview.png", dpi=130, bbox_inches="tight")
print("Preview guardado: weather_shenzhen_preview.png")
plt.show()

In [ ]:
# Guardar CSV
OUT_PATH = "weather_shenzhen.csv"
df_5min.to_csv(OUT_PATH, index=False)

size_kb = Path(OUT_PATH).stat().st_size / 1024
print(f"Guardado: {OUT_PATH}")
print(f"Tamaño:   {size_kb:.1f} KB")
print(f"Filas:    {len(df_5min)}")
print(f"Columnas: {list(df_5min.columns)}")
print(f"\nListo para importar en ChatEV.ipynb (Sección 1.5)")

In [ ]:
# (Opcional) Si estás en Google Colab: descarga el CSV y súbelo a Drive
try:
    from google.colab import files, drive
    # Subir a Drive (mismo directorio que los datasets ST-EVCDP)
    import shutil
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PATH = "/content/drive/MyDrive/MASTER CD/2cuatri/Modelado computacional/Trabajo final/datasets/weather_shenzhen.csv"
    shutil.copy(OUT_PATH, DRIVE_PATH)
    print(f"Copiado a Drive: {DRIVE_PATH}")
    print("Ahora weather_shenzhen.csv está junto al resto de CSVs del dataset.")
    print("ChatEV.ipynb lo leerá automáticamente en el glob de datasets.")
except ImportError:
    print("No estás en Colab — el CSV ya está guardado localmente como weather_shenzhen.csv")
    print("Cópialo manualmente a la carpeta de datasets antes de ejecutar ChatEV.ipynb.")